In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_translation = pd.read_csv('product_category_name_translation.csv')

df_orders['order_purchase_timestamp'] = pd.to_datetime(
    df_orders['order_purchase_timestamp'], errors = 'coerce'
)

In [ ]:
df_products = df_products.merge(
    df_translation, on = 'product_category_name', how = 'left'
)
df_products['product_category_name_english'] = (
    df_products['product_category_name_english'].fillna('unknown')
)

unknown_count = (df_products['product_category_name_english'] == 'unknown').sum()
print(f'товаров без перевода категории: {unknown_count}')

In [ ]:
order_totals = df_items.groupby('order_id').agg(
    items_total = ('price', 'sum'),
    freight_total = ('freight_value', 'sum'),
    items_count = ('order_item_id', 'count')
).reset_index()

order_totals['total_order_value'] = (
    order_totals['items_total'] + order_totals['freight_total']
)
order_totals.head()

In [ ]:
core = (
    df_orders
    .merge(order_totals, on = 'order_id', how = 'left')
    .merge(df_customers, on = 'customer_id', how = 'left')
)

core_delivered = core.loc[core['order_status'] == 'delivered'].copy()

print(f'всего заказов: {len(core)}')
print(f'доставлено: {len(core_delivered)}')
print(f'выручка: {core_delivered["total_order_value"].sum():,.2f} BRL')

In [ ]:
naive_revenue = (
    df_items
    .merge(df_orders[['order_id', 'order_status']], on = 'order_id')
    .query("order_status == 'delivered'")
    .eval('price + freight_value')
    .sum()
)
correct_revenue = core_delivered['total_order_value'].sum()

print(f'корректный расчёт: {correct_revenue:,.2f} BRL')
print(f'наивный расчёт: {naive_revenue:,.2f} BRL')
print(f'задвоение: {naive_revenue - correct_revenue:,.2f} BRL')

In [ ]:
snapshot_date = core_delivered['order_purchase_timestamp'].max() + pd.Timedelta(days = 1)
print(f'дата среза: {snapshot_date}')

rfm = core_delivered.groupby('customer_unique_id').agg(
    recency = ('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id', 'nunique'),
    monetary = ('total_order_value', 'sum')
).reset_index()

rfm.describe().round(2)

In [ ]:
rfm['R'] = pd.qcut(rfm['recency'], 4, labels = [4, 3, 2, 1]).astype(int)
rfm['F'] = pd.cut(
    rfm['frequency'], bins = [0, 1, 2, 5, np.inf], labels = [1, 2, 3, 4]
).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'], 4, labels = [1, 2, 3, 4]).astype(int)

rfm['RFM_score'] = (
    rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
)
rfm.head()

In [ ]:
customer_states = (
    df_customers[['customer_unique_id', 'customer_state']]
    .drop_duplicates(subset = 'customer_unique_id', keep = 'first')
)

champions = (
    rfm.loc[rfm['RFM_score'] == '444']
    .merge(customer_states, on = 'customer_unique_id', how = 'left')
)

print(f'чемпионов: {len(champions)} из {len(rfm)} ({len(champions) / len(rfm):.2%})')
print(champions['customer_state'].value_counts().head(5))

In [ ]:
items_cat = (
    df_items
    .merge(
        df_products[['product_id', 'product_category_name_english']],
        on = 'product_id', how = 'left'
    )
    .merge(
        df_orders[['order_id', 'customer_id', 'order_status']],
        on = 'order_id', how = 'left'
    )
    .merge(
        df_customers[['customer_id', 'customer_state']],
        on = 'customer_id', how = 'left'
    )
    .query("order_status == 'delivered'")
)

top10 = (
    items_cat
    .groupby('product_category_name_english')['price']
    .sum()
    .nlargest(10)
    .index
)

for i, cat in enumerate(top10, 1):
  print(f'{i}. {cat}')

In [ ]:
pivot = (
    items_cat.loc[items_cat['product_category_name_english'].isin(top10)]
    .pivot_table(
        index = 'customer_state',
        columns = 'product_category_name_english',
        values = 'price',
        aggfunc = 'sum',
        fill_value = 0
    )
)

pivot_share = pivot.div(pivot.sum(axis = 1), axis = 0) * 100

print(f'сумма по строке: {pivot_share.sum(axis = 1).round(2).unique()}')
pivot_share.round(2)

In [ ]:
country_avg = pivot_share['health_beauty'].mean()
deviation = (pivot_share['health_beauty'] - country_avg).sort_values(ascending = False)
top_state = deviation.index[0]

print(f'среднее по штатам: {country_avg:.2f}%')
print(f'аномалия: {top_state} — {pivot_share.loc[top_state, "health_beauty"]:.2f}% '
      f'(+{deviation.iloc[0]:.2f} п.п.)')
print(deviation.head(5).round(2))

In [ ]:
cohorts = core_delivered.copy()

cohorts['order_month'] = cohorts['order_purchase_timestamp'].dt.to_period('M')
cohorts['cohort_month'] = (
    cohorts.groupby('customer_unique_id')['order_month'].transform('min')
)
cohorts['month_index'] = (
    (cohorts['order_month'].dt.year - cohorts['cohort_month'].dt.year) * 12
    + (cohorts['order_month'].dt.month - cohorts['cohort_month'].dt.month)
)

cohorts[['customer_unique_id', 'order_month', 'cohort_month', 'month_index']].head()

In [ ]:
cohort_matrix = (
    cohorts
    .groupby(['cohort_month', 'month_index'])['customer_unique_id']
    .nunique()
    .reset_index()
    .pivot_table(
        index = 'cohort_month',
        columns = 'month_index',
        values = 'customer_unique_id'
    )
)

cohort_matrix

In [ ]:
retention = cohort_matrix.div(cohort_matrix.iloc[:, 0], axis = 0) * 100
retention.round(2)